# Adım 3: Spark + Delta Lake — Bronze / Silver / Gold
**Rojda sorumluluğu** — `feature/spark-eda` branch

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month
import os

DATA_PATH   = './data/archive/daily_weather.parquet'
BRONZE_PATH = './delta_lake/bronze'
SILVER_PATH = './delta_lake/silver'
GOLD_PATH   = './delta_lake/gold'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateDataPipeline')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .config('spark.sql.parquet.datetimeRebaseModeInRead', 'CORRECTED')
        .config('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
        .getOrCreate()
    )

def write_bronze(spark):
    print('[BRONZE] Ham veri okunuyor...')
    df = spark.read.parquet(DATA_PATH)
    count = df.count()
    print(f'[BRONZE] Kayit sayisi: {count:,}')
    df.write.format('delta').mode('overwrite').save(BRONZE_PATH)
    print(f'[BRONZE] Yazildi -> {BRONZE_PATH}')
    return df

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
write_bronze(spark)

26/05/12 14:47:36 WARN Utils: Your hostname, Canpolat-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/05/12 14:47:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/canpolat/.ivy2/cache
The jars for the packages stored in: /Users/canpolat/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ec013b43-ba61-43ed-afde-7ba7c2d996dc;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 75ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	------------------------------------------------------------------

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/05/12 14:47:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


[BRONZE] Ham veri okunuyor...
[BRONZE] Kayit sayisi: 27,635,763


26/05/12 14:48:11 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE] Yazildi -> ./delta_lake/bronze


DataFrame[station_id: string, city_name: string, date: timestamp_ntz, season: string, avg_temp_c: double, min_temp_c: double, max_temp_c: double, precipitation_mm: double, snow_depth_mm: double, avg_wind_dir_deg: double, avg_wind_speed_kmh: double, peak_wind_gust_kmh: double, avg_sea_level_pres_hpa: double, sunshine_total_min: double, __index_level_0__: bigint]

## Silver Katmanı
Null temizleme ve duplike kaldırma

In [2]:
def write_silver(spark):
    print('[SILVER] Bronze okunuyor, temizleniyor...')
    df = spark.read.format('delta').load(BRONZE_PATH)
    df_clean = df.dropna(subset=['avg_temp_c', 'station_id', 'date'])
    before = df.count()
    df_clean = df_clean.dropDuplicates(['station_id', 'date'])
    after = df_clean.count()
    print(f'[SILVER] Ham: {before:,}  ->  Temiz: {after:,}  ({before - after:,} satir kaldirildi)')
    df_clean.write.format('delta').mode('overwrite').save(SILVER_PATH)
    print(f'[SILVER] Yazildi -> {SILVER_PATH}')
    return df_clean

write_silver(spark)

[SILVER] Bronze okunuyor, temizleniyor...


[SILVER] Ham: 27,635,763  ->  Temiz: 21,112,400  (6,523,363 satir kaldirildi)


[50.006s][warning][gc,alloc] Executor task launch worker for task 7.0 in stage 37.0 (TID 357): Retried waiting for GCLocker too often allocating 262144 words


26/05/12 14:48:26 WARN TaskMemoryManager: Failed to allocate a page (2097136 bytes), try again.
26/05/12 14:48:45 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/05/12 14:48:54 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


[SILVER] Yazildi -> ./delta_lake/silver


DataFrame[station_id: string, city_name: string, date: timestamp_ntz, season: string, avg_temp_c: double, min_temp_c: double, max_temp_c: double, precipitation_mm: double, snow_depth_mm: double, avg_wind_dir_deg: double, avg_wind_speed_kmh: double, peak_wind_gust_kmh: double, avg_sea_level_pres_hpa: double, sunshine_total_min: double, __index_level_0__: bigint]

## Gold Katmanı
Fiziksel filtreler + year/month sütunları ekleniyor

In [3]:
def write_gold(spark):
    print('[GOLD] Silver okunuyor, ozellikler ekleniyor...')
    df = spark.read.format('delta').load(SILVER_PATH)
    df_gold = (
        df
        .filter(col('avg_temp_c').between(-60, 60))
        .filter(col('min_temp_c') <= col('avg_temp_c'))
        .filter(col('avg_temp_c') <= col('max_temp_c'))
        .select(
            col('station_id'), col('city_name'), col('date'), col('season'),
            col('avg_temp_c'), col('min_temp_c'), col('max_temp_c'),
            col('precipitation_mm'), col('snow_depth_mm'),
            col('avg_wind_speed_kmh'), col('avg_sea_level_pres_hpa'),
            col('sunshine_total_min'),
            year(col('date')).alias('year'),
            month(col('date')).alias('month'),
        )
    )
    count = df_gold.count()
    print(f'[GOLD] Kayit sayisi: {count:,}')
    df_gold.write.format('delta').mode('overwrite').save(GOLD_PATH)
    print(f'[GOLD] Yazildi -> {GOLD_PATH}')
    return df_gold

df_gold = write_gold(spark)
print(f'Bronze -> {BRONZE_PATH}')
print(f'Silver -> {SILVER_PATH}')
print(f'Gold   -> {GOLD_PATH}')
df_gold.show(5, truncate=False)
spark.stop()
print('Pipeline tamamlandi.')

[GOLD] Silver okunuyor, ozellikler ekleniyor...


[GOLD] Kayit sayisi: 15,882,533


26/05/12 14:49:05 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/05/12 14:49:10 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


[GOLD] Yazildi -> ./delta_lake/gold
Bronze -> ./delta_lake/bronze
Silver -> ./delta_lake/silver
Gold   -> ./delta_lake/gold
+----------+------------+-------------------+------+----------+----------+----------+----------------+-------------+------------------+----------------------+------------------+----+-----+
|station_id|city_name   |date               |season|avg_temp_c|min_temp_c|max_temp_c|precipitation_mm|snow_depth_mm|avg_wind_speed_kmh|avg_sea_level_pres_hpa|sunshine_total_min|year|month|
+----------+------------+-------------------+------+----------+----------+----------+----------------+-------------+------------------+----------------------+------------------+----+-----+
|01008     |Longyearbyen|1975-10-23 00:00:00|Autumn|-5.1      |-7.4      |-2.5      |1.0             |null         |null              |null                  |null              |1975|10   |
|01008     |Longyearbyen|1975-12-17 00:00:00|Winter|-20.9     |-23.3     |-17.9     |0.0             |null         |null